In [ ]:
# Day 2 pipeline: feature selection, SMOTE, XGBoost tuning, improved NN
import os, glob
import numpy as np
import pandas as pd
from helper_functions import *
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import randint, uniform

In [ ]:
# SETTINGS 
DATA_DIR = "data"
CSV_GLOB = os.path.join(DATA_DIR, "*.csv")
LABEL_COL = "Label"
RANDOM_STATE = 42
TOP_K = 50   # for mutual information selector
SMOTE_KNN = 5

In [ ]:
#  LOAD DATA 
csvs = sorted(glob.glob(CSV_GLOB))
if not csvs:
    raise FileNotFoundError("No CSVs found in data/ — download CICDDoS2019 CSV(s) into data/")
df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True) if len(csvs)>1 else pd.read_csv(csvs[0])
print("Loaded shape:", df.shape)

In [ ]:
# basic clean (reuse your previous function)
df = basic_cleaning(df, drop_cols=['Flow ID','Timestamp','Source IP','Destination IP'])
df = df.replace([np.inf, -np.inf], np.nan)
df = simple_impute_numeric(df, strategy='median')

In [ ]:
# detect label col if different
if LABEL_COL not in df.columns:
    possible = [c for c in df.columns if 'label' in c.lower() or 'attack' in c.lower() or 'category' in c.lower()]
    if possible:
        LABEL_COL = possible[0]
        print("Using label column:", LABEL_COL)
    else:
        raise ValueError("No label column detected")

In [ ]:
# SELECT numeric features
X = df.select_dtypes(include=[np.number]).drop(columns=[LABEL_COL], errors='ignore')
y = df[LABEL_COL].astype(str)   # treat labels as strings for selector


In [ ]:
#  reduce to top-N features by mutual info
sel_k, mi_scores, mi_selected = select_kbest_mutual_info(X, y, k=TOP_K, random_state=RANDOM_STATE)
print("Top mutual-info features:", mi_selected[:10])

In [ ]:
#  get tree-based importances and intersect
tree_selector, tree_importances, tree_selected = select_from_tree(X[mi_selected], y, threshold='median', random_state=RANDOM_STATE)
print(f"Selected {len(tree_selected)} features by tree importance (threshold median).")


In [ ]:
# Final feature set = intersection of both (conservative)
final_features = sorted(list(set(mi_selected).intersection(set(tree_selected))))
if len(final_features) < 5:
    # fallback: use top mi_selected
    final_features = mi_selected[:min(30, len(mi_selected))]
print("Final feature count:", len(final_features))


In [ ]:
# plot importances
plot_feature_importances(tree_importances, top_n=30, title="Tree-based importances (subset)")


In [ ]:

# Train/test split
X_sel = X[final_features].copy()
X_train, X_test, y_train, y_test = train_test_split(X_sel, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


In [ ]:
#  scale numeric features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
joblib.dump(scaler, "scaler_day2.joblib")


In [ ]:
#  Handle imbalance (SMOTE) 
print("Original train class distribution:\n", pd.Series(y_train).value_counts())
X_train_bal, y_train_bal = apply_smote(X_train_s, y_train, sampling_strategy='auto', random_state=RANDOM_STATE, k_neighbors=SMOTE_KNN)
print("After SMOTE class distribution:\n", pd.Series(y_train_bal).value_counts())


In [ ]:
# XGBoost baseline + hyperparameter tuning
xgb_clf = xgb.XGBClassifier(objective='multi:softprob' if len(y.unique())>2 else 'binary:logistic',
                            eval_metric='mlogloss' if len(y.unique())>2 else 'logloss',
                            use_label_encoder=False,
                            tree_method='hist',
                            random_state=RANDOM_STATE,
                            n_jobs=-1)


In [ ]:
param_dist = {
    'n_estimators': randint(100, 800),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 2)
}

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
rand_search = RandomizedSearchCV(xgb_clf, param_distributions=param_dist,
                                 n_iter=30, scoring='f1_macro' if len(y.unique())>2 else 'f1',
                                 n_jobs=-1, cv=cv, verbose=2, random_state=RANDOM_STATE)
rand_search.fit(X_train_bal, y_train_bal)
print("Best params:", rand_search.best_params_)
best_xgb = rand_search.best_estimator_

In [ ]:
# Evaluate XGBoost
y_pred_xgb = best_xgb.predict(X_test_s)
print("XGBoost accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb, digits=4))
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6,5)); sns.heatmap(cm, annot=True, fmt='d'); plt.title("XGBoost Confusion Matrix"); plt.show()


In [ ]:
# Save model
joblib.dump(best_xgb, "xgb_day2_best.joblib")

In [ ]:
#  compute multiclass AUC (if probabilities available)
if hasattr(best_xgb, "predict_proba"):
    y_proba_xgb = best_xgb.predict_proba(X_test_s)
    aucs, avg_auc = compute_multiclass_auc(pd.factorize(y_test)[0], y_proba_xgb)
    if aucs is not None:
        print("Per-class AUCs:\n", aucs)
        print("Average AUC:", avg_auc)


In [ ]:
# Improved Neural Network (Keras) 
# Use class weights OR SMOTE-ed data. Here we'll use class weights with original X_train_s to keep variety.
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {i: w for i,w in enumerate(class_weights)}
print("Class weights (mapping):", class_weight_dict)


In [ ]:
# Keras needs integer labels starting at 0
label_to_int = {lab:i for i,lab in enumerate(sorted(y.unique()))}
y_train_int = y_train.map(label_to_int).astype(int)
y_test_int = y_test.map(label_to_int).astype(int)


In [ ]:
# Use scaled training set (not SMOTE here); you can also train on X_train_bal if desired.
num_classes = len(label_to_int)
y_train_enc = keras.utils.to_categorical(y_train_int, num_classes=num_classes)
y_test_enc = keras.utils.to_categorical(y_test_int, num_classes=num_classes)



In [ ]:
def focal_loss(gamma=2., alpha=.25):
    import tensorflow as tf
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return tf.reduce_sum(loss, axis=1)
    return loss

In [ ]:
# Build model
from tensorflow.keras import layers, models, optimizers, callbacks
def build_dense_model(input_dim, num_classes, dropout=0.3):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout/2),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

In [ ]:
model = build_dense_model(X_train_s.shape[1], num_classes, dropout=0.3)
model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss=focal_loss(gamma=2., alpha=.25),
              metrics=['accuracy'])
model.summary()